In [1]:
import pandas as pd
from pathlib import Path
import re

## Conversion Functions

In [10]:
def find_data_start_line_and_header(filepath):
    """
    Find the line number where measurement data starts and extract the header line as a list of column names.
    Returns (data_start_line, encoding, header_cols)
    """
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            with open(filepath, 'r', encoding=encoding) as f:
                for i, line in enumerate(f):
                    if line.strip().startswith('Date [A Ch.1 Main]'):
                        header_line = line.strip()
                        header_cols = [col.strip() for col in header_line.split('\t')]
                        return i + 1, encoding, header_cols
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode or find data start line in {filepath}")


def find_column(columns, *keywords):
    def normalize(s):
        return ' '.join(s.lower().split())
    norm_cols = [normalize(col) for col in columns]
    for col, norm_col in zip(columns, norm_cols):
        if all(kw.lower() in norm_col for kw in keywords):
            return col
    print("Available columns:")
    for col in columns:
        print(f"  '{col}'")
    raise ValueError(f"Could not find column with keywords: {keywords}")


def parse_newbox_txt(filepath):
    """
    Parse PyroScience Workbench .txt file and extract key columns.
    Returns DataFrame with columns: seconds, hours, clock, Ch1, Ch2, Ch3, Ch4, Temp
    """
    data_start_line, encoding, header_cols = find_data_start_line_and_header(filepath)
    print(f"  Using encoding: {encoding}")
    print(f"  Using header columns: {header_cols[:8]} ... (total {len(header_cols)})")
    df = pd.read_csv(
        filepath,
        sep='\t',
        skiprows=data_start_line,
        names=header_cols,
        encoding=encoding,
        low_memory=False
    )
    print("Read columns:")
    for col in df.columns:
        print(f"  '{col}'")
    result = pd.DataFrame()

    # Find columns robustly
    seconds_col = find_column(df.columns, 'dt (s)', 'ch.1')
    result['seconds'] = df[seconds_col]
    result['hours'] = result['seconds'] / 3600.0

    time_col = find_column(df.columns, 'time', 'ch.1')
    result['clock'] = df[time_col]

    for ch_num in [1, 2, 3, 4]:
        oxygen_col = find_column(df.columns, 'oxygen', 'ch.' + str(ch_num))
        result[f'Ch{ch_num}'] = df[oxygen_col]

    temp_col = find_column(df.columns, 'temp', 'ch.1')
    result['Temp'] = df[temp_col]

    return result


def convert_newbox_file(input_path, output_path=None, output_dir=None):
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")
    if output_path:
        output_path = Path(output_path)
    else:
        stem = input_path.stem
        match = re.search(r'(Ostracods-trial\d+(?:\.\d+)?-newbox-\d+\w+)', stem)
        if match:
            base_name = match.group(1)
        else:
            base_name = re.sub(r'^\d{4}-\d{2}-\d{2}_\d{6}_', '', stem)
        if output_dir:
            output_path = Path(output_dir) / f"{base_name}.csv"
        else:
            output_path = input_path.parent / f"{base_name}.csv"
    print(f"Converting: {input_path.name}")
    print(f"Output to: {output_path}")
    df = parse_newbox_txt(input_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"✓ Converted {len(df)} rows")
    print(f"  Time range: {df['hours'].min():.2f} - {df['hours'].max():.2f} hours")
    print(f"  Temp range: {df['Temp'].min():.2f} - {df['Temp'].max():.2f} °C")
    return output_path


## Convert Trial 4 Newbox (Test Case)

In [11]:
# Path to trial4 newbox file
input_file = Path('../data/original_respirometry_Panama2025/Ostracods-trial4-newbox-13Nov/2025-11-13_185203_Ostracods-trial4-newbox-13Nov/2025-11-13_185203_Ostracods-trial4-newbox-13Nov.txt')
output_dir = Path('../data/original_respirometry_Panama2025')

# Convert
output_file = convert_newbox_file(input_file, output_dir=output_dir)
print(f"\n✓ Success! Output saved to: {output_file}")

Converting: 2025-11-13_185203_Ostracods-trial4-newbox-13Nov.txt
Output to: ../data/original_respirometry_Panama2025/Ostracods-trial4-newbox-13Nov.csv
  Using encoding: latin-1
  Using header columns: ['Date [A Ch.1 Main]', 'Time [A Ch.1 Main]', 'dt (s) [A Ch.1 Main]', 'Oxygen (µmol/L) [A Ch.1 Main]', 'dphi (°) [A Ch.1 Main]', 'Signal Intensity (mV) [A Ch.1 Main]', 'Ambient Light (mV) [A Ch.1 Main]', 'Status [A Ch.1 Main]'] ... (total 80)
Read columns:
  'Date [A Ch.1 Main]'
  'Time [A Ch.1 Main]'
  'dt (s) [A Ch.1 Main]'
  'Oxygen (µmol/L) [A Ch.1 Main]'
  'dphi (°) [A Ch.1 Main]'
  'Signal Intensity (mV) [A Ch.1 Main]'
  'Ambient Light (mV) [A Ch.1 Main]'
  'Status [A Ch.1 Main]'
  'Date [A Ch.1 CompT]'
  'Time [A Ch.1 CompT]'
  'dt (s) [A Ch.1 CompT]'
  'Sample Temp. (°C) [A Ch.1 CompT]'
  'Status [A Ch.1 CompT]'
  'Date [A Ch.1 CompP]'
  'Time [A Ch.1 CompP]'
  'dt (s) [A Ch.1 CompP]'
  'Pressure (mbar) [A Ch.1 CompP]'
  'Status [A Ch.1 CompP]'
  'Date [A Ch.2 Main]'
  'Time [A Ch.2

## Verify the Output

In [12]:
# Load and preview the converted file
df_converted = pd.read_csv(output_file)
print(f"Shape: {df_converted.shape}")
print(f"\nColumns: {list(df_converted.columns)}")
print(f"\nFirst few rows:")
df_converted.head(10)

Shape: (9017, 8)

Columns: ['seconds', 'hours', 'clock', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Temp']

First few rows:


,seconds,hours,clock,Ch1,Ch2,Ch3,Ch4,Temp
0,0.929,0.000258,18:52:04.389,192.320663,185.410049,181.501144,186.224182,27.010
1,5.926,0.001646,18:52:09.387,193.194946,185.318253,181.991455,187.390396,27.008
2,10.928,0.003036,18:52:14.388,191.825867,185.176010,183.390396,186.650085,27.015
3,15.927,0.004424,18:52:19.388,193.265961,185.881363,181.507278,187.457001,27.017
4,20.929,0.005814,18:52:24.389,191.859665,183.683533,183.416748,186.241089,27.016
5,25.928,0.007202,18:52:29.388,192.451126,184.015518,182.260269,187.023163,27.015
6,30.929,0.008591,18:52:34.389,193.650330,185.404999,183.562103,188.511230,27.019
7,35.930,0.009981,18:52:39.390,192.299774,183.168900,182.924332,186.841553,27.014
8,40.930,0.011369,18:52:44.390,194.637573,184.922226,182.682724,185.665237,27.020
9,45.930,0.012758,18:52:49.390,191.581741,184.900162,182.697617,186.533325,27.017


In [13]:
# Quick statistics
df_converted.describe()

,seconds,hours,Ch1,Ch2,Ch3,Ch4,Temp
count,9017.000000,9017.000000,9017.000000,9017.000000,9017.000000,9017.000000,9017.000000
mean,22540.754520,6.261321,163.557639,160.650819,158.146708,164.418425,27.156650
std,13016.042681,3.615567,13.237103,12.785716,12.078765,11.575444,0.039629
min,0.929000,0.000258,140.167557,137.594025,136.998444,144.000458,27.008000
25%,11270.372000,3.130659,151.934738,149.648651,148.063461,154.506531,27.139000
50%,22540.891000,6.261359,163.212509,159.981445,157.216003,163.595825,27.167000
75%,33811.433000,9.392065,174.735901,171.424591,167.800476,173.728165,27.186000
max,45081.180000,12.522550,194.637573,186.130981,183.623428,188.511230,27.206000


## Batch Convert All Newbox Files

Once trial4 looks good, convert all remaining newbox files (trials 4.5, 5, 5.5, 6, 6.5, 7, 7.5).

In [15]:
# Define all newbox files to convert
# Use absolute paths for clarity and to avoid path issues
newbox_files = [
    # Trial 5
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial5-newbox-14Nov/2025-11-14_184602_Ostracods-trial5-newbox-14Nov/2025-11-14_184602_Ostracods-trial5-newbox-14Nov.txt',
    # Trial 5.5
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial5-light-newbox-15Nov/2025-11-15_083727_Ostracods-trial5-light-newbox-/2025-11-15_083727_Ostracods-trial5-light-newbox-.txt',
    # Trial 6 (day)
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial6-light-newbox-16Nov/2025-11-16_083516_Ostracods-trial6-light-newbox-/2025-11-16_083516_Ostracods-trial6-light-newbox-.txt',
    # Trial 6.5 (night)
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial6-dark-newbox-16Nov/2025-11-16_180908_Ostracods-trial6-dark-newbox-1/2025-11-16_180908_Ostracods-trial6-dark-newbox-1.txt',
    # Trial 7 (day)
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial7-light-newbox-17Nov/2025-11-17_082644_Ostracods-trial7-light-newbox-/2025-11-17_082644_Ostracods-trial7-light-newbox-.txt',
    # Trial 7.5 (night)
    '/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial7-dark-newbox-17Nov/2025-11-17_185851_Ostracods-trial7-dark-newbox-1/2025-11-17_185851_Ostracods-trial7-dark-newbox-1.txt',
]

data_dir = Path('/Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025')

print(f"Converting {len(newbox_files)} newbox files...\n")

converted_files = []
for file_path in newbox_files:
    full_path = Path(file_path)
    try:
        output = convert_newbox_file(full_path, output_dir=data_dir)
        converted_files.append(output)
        print()
    except Exception as e:
        print(f"✗ Error converting {file_path}: {e}\n")

print(f"\n{'='*60}")
print(f"✓ Batch conversion complete! Converted {len(converted_files)} files.")
print(f"{'='*60}")


Converting 6 newbox files...

Converting: 2025-11-14_184602_Ostracods-trial5-newbox-14Nov.txt
Output to: /Users/oakley/Documents/GitHub/signal_respirometry/data/original_respirometry_Panama2025/Ostracods-trial5-newbox-14Nov.csv
  Using encoding: latin-1
  Using header columns: ['Date [A Ch.1 Main]', 'Time [A Ch.1 Main]', 'dt (s) [A Ch.1 Main]', 'Oxygen (µmol/L) [A Ch.1 Main]', 'dphi (°) [A Ch.1 Main]', 'Signal Intensity (mV) [A Ch.1 Main]', 'Ambient Light (mV) [A Ch.1 Main]', 'Status [A Ch.1 Main]'] ... (total 80)
Read columns:
  'Date [A Ch.1 Main]'
  'Time [A Ch.1 Main]'
  'dt (s) [A Ch.1 Main]'
  'Oxygen (µmol/L) [A Ch.1 Main]'
  'dphi (°) [A Ch.1 Main]'
  'Signal Intensity (mV) [A Ch.1 Main]'
  'Ambient Light (mV) [A Ch.1 Main]'
  'Status [A Ch.1 Main]'
  'Date [A Ch.1 CompT]'
  'Time [A Ch.1 CompT]'
  'dt (s) [A Ch.1 CompT]'
  'Sample Temp. (°C) [A Ch.1 CompT]'
  'Status [A Ch.1 CompT]'
  'Date [A Ch.1 CompP]'
  'Time [A Ch.1 CompP]'
  'dt (s) [A Ch.1 CompP]'
  'Pressure (mbar) [A

## Summary of Converted Files

In [16]:
# List all converted CSV files
print("Converted files:")
for f in converted_files:
    print(f"  - {f.name}")

Converted files:
  - Ostracods-trial5-newbox-14Nov.csv
  - Ostracods-trial5-light-newbox-.csv
  - Ostracods-trial6-light-newbox-.csv
  - Ostracods-trial6-dark-newbox-1.csv
  - Ostracods-trial7-light-newbox-.csv
  - Ostracods-trial7-dark-newbox-1.csv
